In [1]:
import os
import numpy as np
import pandas as pd

try:
    import tensorflow as tf
except BaseException:  # catch ImportError, protobuf errors, etc.
    tf = None


def locate_file(filename: str) -> str:
    """
    Return the first existing full path for `filename` searching common
    Kaggle data locations. Raises FileNotFoundError if none are found.
    """
    candidates = [
        os.path.join("data", "hms-harmful-brain-activity-classification", filename),
        os.path.join("data", filename),
        os.path.join("input", "hms-harmful-brain-activity-classification", filename),
        os.path.join("input", filename),
        os.path.join(
            "/kaggle/input", "hms-harmful-brain-activity-classification", filename
        ),
        os.path.join("/kaggle/input", filename),
        os.path.join("/kaggle/working", filename),
        filename,  # fallback: current working directory
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Unable to locate {filename} in any known location.")


SPLITS = 5  # Number of folds expected in the original script.
BATCHSIZE = 32
DATATYPE = []  # Empty list forces the fallback path (no model usage).
TARGETS = None  # Will be set after loading the train data.




2026-05-05 05:38:04.568321: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777959484.581405      57 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777959484.585455      57 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-05 05:38:04.601329: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
def _simple_backbone(input_tensor):
    """A lightweight convolution‑pooling block used instead of EfficientNet."""
    if tf is None:
        raise RuntimeError("TensorFlow is required for the backbone.")
    x = tf.keras.layers.Conv2D(16, (3, 3), padding="same", activation="relu")(
        input_tensor
    )
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    return x


def build_model():
    """Build the model only if TensorFlow is available; otherwise return None."""
    if tf is None:
        return None
    inp = []
    if "spe" in DATATYPE:
        inp_spe = tf.keras.Input(shape=(4, 0, 0))  # placeholder shapes
        x_spe = _simple_backbone(inp_spe)
        inp.append(inp_spe)
        y = x_spe
    if "eeg" in DATATYPE:
        inp_eeg = tf.keras.Input(shape=(0, 0))
        x_eeg = _simple_backbone(inp_eeg)
        inp.append(inp_eeg)
        y = (
            tf.keras.layers.Concatenate(axis=1)([y, x_eeg])
            if "spe" in DATATYPE
            else x_eeg
        )
    if "stft" in DATATYPE:
        inp_stft = tf.keras.Input(shape=(0,))
        x_stft = _simple_backbone(inp_stft)
        inp.append(inp_stft)
        y = (
            tf.keras.layers.Concatenate(axis=1)([y, x_stft])
            if any(t in DATATYPE for t in ["spe", "eeg"])
            else x_stft
        )
    if "img" in DATATYPE:
        inp_img = tf.keras.Input(shape=(0, 0, 3))
        x_img = _simple_backbone(inp_img)
        inp.append(inp_img)
        y = (
            tf.keras.layers.Concatenate(axis=1)([y, x_img])
            if any(t in DATATYPE for t in ["spe", "eeg", "stft"])
            else x_img
        )
    if (
        "spe" in DATATYPE
        or "eeg" in DATATYPE
        or "stft" in DATATYPE
        or "img" in DATATYPE
    ):
        y = tf.keras.layers.Dense(len(TARGETS), activation="softmax", dtype="float32")(
            y
        )
        return tf.keras.Model(inputs=inp, outputs=y)
    return None


class DataGenerator:
    """Placeholder class used when TensorFlow is unavailable."""

    pass




In [3]:
train_path = locate_file("train.csv")
df = pd.read_csv(train_path)
TARGETS = df.columns[-6:]  # seizure, lpd, gpd, lrda, grda, other
print("Train shape:", df.shape)
print("Targets:", list(TARGETS))

test_path = locate_file("test.csv")
test = pd.read_csv(test_path)
print("Test shape:", test.shape)

# Compute overall class prior
overall_prior = df[TARGETS].sum()
overall_prior = overall_prior / overall_prior.sum()
overall_arr = overall_prior.values.astype(np.float32)

# Patient‑level priors (row‑wise normalized)
patient_prior_df = (
    df.groupby("patient_id")[TARGETS]
    .sum()
    .apply(lambda x: x / x.sum() if x.sum() != 0 else overall_prior)
)
patient_prior_df = patient_prior_df.reindex(columns=TARGETS)

# EEG‑level priors (row‑wise normalized)
eeg_prior_df = (
    df.groupby("eeg_id")[TARGETS]
    .sum()
    .apply(lambda x: x / x.sum() if x.sum() != 0 else overall_prior)
)
eeg_prior_df = eeg_prior_df.reindex(columns=TARGETS)

# Use pure specific prior (no blending)
alpha = 1.0

pred_list = []
for _, row in test.iterrows():
    pid = row["patient_id"]
    eid = row["eeg_id"]

    if eid in eeg_prior_df.index:
        specific_arr = eeg_prior_df.loc[eid].values.astype(np.float32)
    elif pid in patient_prior_df.index:
        specific_arr = patient_prior_df.loc[pid].values.astype(np.float32)
    else:
        specific_arr = overall_arr

    # With alpha=1.0 this is just the specific prior
    blended = alpha * specific_arr + (1 - alpha) * overall_arr
    pred_list.append(blended)

pred = np.stack(pred_list, axis=0)

# Ensure each row sums to 1 (numeric safety)
pred = pred / pred.sum(axis=1, keepdims=True)

sub = pd.DataFrame({"eeg_id": test["eeg_id"].values})
sub[TARGETS] = pred
submission_path = "submission.csv"
sub.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path} with shape:", sub.shape)

Train shape: (96950, 15)
Targets: ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
Test shape: (9850, 3)
Submission written to submission.csv with shape: (9850, 7)
